# 문항 1 금융위 주식시세정보 API 호출과 오류 처리

In [ ]:
import os
import requests
from dotenv import load_dotenv
from urllib.parse import unquote, quote
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd

today = datetime.now().strftime("%Y%m%d")
ten_days_ago = (datetime.now() - timedelta(days=10)).strftime("%Y%m%d")


load_dotenv()

API_KEY = os.getenv('OPENAI_API_KEY')
if not API_KEY:
    raise ValueError("인증키가 없습니다.")

URL = 'https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo'

PARAMS = {
    'serviceKey' : unquote(API_KEY),
    'resultType' : 'json',
    'beginBasDt' : ten_days_ago
}

class OpenAPIKeyError(Exception): ...

class OpenAPIError(Exception): ...

def fetch(codes):
    result = []
    for code in codes:
        response = requests.get(URL, params={**PARAMS, 'likeSrtnCd': code}, timeout=10)
        response.raise_for_status()

        data = response.json()

        resultMsg = data['response']['header']['resultMsg']
        if resultMsg == 'SERVICE_KEY_IS_NOT_REGISTERED_ERROR':
            raise OpenAPIKeyError("인증키 오류")
        elif resultMsg == 'LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR':
            raise OpenAPIError("일일 쿼터 초과")
        elif resultMsg == 'INVALID_REQUEST_PARAMETER_ERROR':
            raise OpenAPIError("필수 파라미터 누락")
        if len(result) == 0:
            result = data['response']['body']['items']['item']
        else:
            for item in data['response']['body']['items']['item']:
                result.append(item)
    return result


data = fetch(['005930'])
data_five_days = data[:5]  # 최근 5일치 데이터 확인

df = pd.DataFrame(data_five_days)
df

,basDt,srtnCd,isinCd,itmsNm,mrktCtg,clpr,vs,fltRt,mkp,hipr,lopr,trqu,trPrc,lstgStCnt,mrktTotAmt
0,20260826,005930,KR7005930003,삼성전자,KOSPI,261500,4500,1.75,256500,266500,255500,19532523,5109878328250,5846278608,1528801855992000
1,20260825,005930,KR7005930003,삼성전자,KOSPI,257000,0,0,249000,258000,245000,21617407,5445381811750,5846278608,1502493602256000
2,20260824,005930,KR7005930003,삼성전자,KOSPI,257000,-24500,-8.7,271500,272000,255000,32451940,8456072427750,5846278608,1502493602256000
3,20260821,005930,KR7005930003,삼성전자,KOSPI,281500,10500,3.87,267000,285000,266000,27746471,7703213942500,5846278608,1645727428152000
4,20260820,005930,KR7005930003,삼성전자,KOSPI,271000,23500,9.49,257000,273000,252500,26095919,6961393123500,5846278608,1584341502768000


* .gitignore

.env

* .env .example

OPENAI_API_KEY=(key)

# 문항 2 DB 스키마 설계와 적재 검증

In [ ]:
CREATE DATABASE IF NOT EXISTS fsc_db DEFAULT CHARACTER SET utf8mb4;
USE fsc_db;

CREATE TABLE IF NOT EXISTS raw_item (
    raw_id BIGINT AUTO_INCREMENT PRIMARY KEY,
    source VARCHAR(100) NOT NULL,
    url VARCHAR(255) NOT NULL,
    collected_at DATETIME NOT NULL,
    payload TEXT NOT NULL,
    content_hash CHAR(64) NOT NULL UNIQUE,
    KEY idx_source_collected_at (source, collected_at)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;

In [25]:
import os
import requests
from dotenv import load_dotenv
from urllib.parse import unquote, quote
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import hashlib
import pymysql
import warnings
warnings.filterwarnings("ignore")

class OpenAPIKeyError(Exception): ...

class OpenAPIError(Exception): ...

start_date = datetime.strptime("2025-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2025-12-31", "%Y-%m-%d")

load_dotenv()

API_KEY = os.getenv('OPENAI_API_KEY')
if not API_KEY:
    raise OpenAPIKeyError("인증키가 없습니다.")

URL = 'https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo'

PARAMS = {
    'serviceKey' : unquote(API_KEY),
    'resultType' : 'json',
    'beginBasDt' : start_date.strftime("%Y%m%d"),
    'endBasDt' : end_date.strftime("%Y%m%d"),
}

BATCH = 500

SOURCE='fsc_api'

CODES = ["005930", "000660", "035420", "051910", "005380",
         "006400", "035720", "068270", "105560", "055550"]

def fetch(codes):
    result = []
    for code in codes:
        response = requests.get(URL, params={**PARAMS, 'likeSrtnCd': code}, timeout=10)
        response.raise_for_status()

        data = response.json()

        resultMsg = data['response']['header']['resultMsg']
        if resultMsg == 'SERVICE_KEY_IS_NOT_REGISTERED_ERROR':
            raise OpenAPIKeyError("인증키 오류")
        elif resultMsg == 'LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR':
            raise OpenAPIError("일일 쿼터 초과")
        elif resultMsg == 'INVALID_REQUEST_PARAMETER_ERROR':
            raise OpenAPIError("필수 파라미터 누락")
        if len(result) == 0:
            result = data['response']['body']['items']['item']
        else:
            for item in data['response']['body']['items']['item']:
                result.append(item)
    return result

def collect_data():
    data = fetch(CODES)
    collected_at = datetime.now().strftime("%Y-%m-%d")

    df = pd.DataFrame(data)

    data_dict = df.to_dict(orient='records')

    data_sql = []
    for row in data_dict:
        payload = str(row)
        # 수집소스 | 날짜 | 주식코드
        to_be_hashed = "|".join([SOURCE, row['basDt'], row['srtnCd']])
        content_hash = hashlib.sha256(to_be_hashed.encode()).hexdigest()
        data_sql.append([SOURCE, URL, collected_at, payload, content_hash])

    return data_sql

def connect():
    conn = pymysql.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT')),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        database=os.getenv('DB_NAME'),
        charset='utf8mb4'
    )
    return conn

def insert_data(data):
    conn = connect()
    cursor = conn.cursor()
    insert_query = """
    INSERT INTO raw_item (source, url, collected_at, payload, content_hash)
    VALUES (%s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        collected_at = VALUES(collected_at),
        payload = VALUES(payload)
    """
    for i in range(0, len(data), BATCH):
        with conn.cursor() as cur:
            cur.executemany(insert_query, data[i:i+BATCH])
        conn.commit()
        print("적재%d / %d" % (min(i + BATCH, len(data)), len(data)))
    return None

def dupe_checker():
    query = """
    SELECT source, COUNT(*) AS 행수
    FROM raw_item
    GROUP BY source
    """
    conn = connect()
    try:
        return pd.read_sql(query, conn)
    finally:
        conn.close()

data = collect_data()

# 첫번째
insert_data(data)
print(dupe_checker(), "\n")

# 두번째
insert_data(data)
print(dupe_checker())

적재100 / 100
    source   행수
0  fsc_api  100 

적재100 / 100
    source   행수
0  fsc_api  100


* .env

OPENAI_API_KEY=(key)



DB_HOST=localhost

DB_PORT=3306

DB_USER=root

DB_PASSWORD=test1234

DB_NAME=fsc_db
